In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback
import os

from pyspark.sql.functions import col, when, lit, month, year, concat, left, expr

from typing import Any
 
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    col, when, lit, substring, concat_ws, coalesce, length,max,concat,format_string,expr,regexp_replace,cast, date_format
)

from typing import Optional
 
import delta  # type: ignore
from pyspark.sql import SparkSession
 
from io import BytesIO
import pandas as pd
import json
import sys
 
from datetime import datetime
from pytz import timezone
 
 
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


"""
Accessing the Hive metastore requires specific considerations:
 
1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.
 
2. To access both Unity catalog and AWS services, an access token from Amazon is required.
 
3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html
 
4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.
 
5. After you authenticate successfully querying the Hive metastore is also working.
"""
 
try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared"
 
# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
   
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
 
secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)
 
aws_key = aws_keys[0] if aws_keys else None
 
# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
 
if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                          
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')
 
    S3_client = boto3.client('s3', region_name='eu-central-1')                          
 
 
# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']
 
# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')
redis_link = connect_to_aws_and_access_ssm('/cfn/redis/primary/address')
print(f"env_prefix : {env_prefix}")

In [0]:
# Create a DataFrame with all parameters
env = env_prefix.lower()
if env == 'p':
    environment = 'prod'
elif env == 'qa':
    environment = 'qa'
elif env == 'd':
    environment = 'dev'
else:
    raise ValueError("Invalid environment")

params = {
    "environment": environment,
    "report_category": dbutils.widgets.get("report_category"),
    "report_name": dbutils.widgets.get("report_name"),
    "portfolio_report": dbutils.widgets.get("portfolio_report"),
    "job_id": dbutils.widgets.get("job_id"),
    "delivery_id": dbutils.widgets.get("delivery_id"),
    "email_id": dbutils.widgets.get("email_id"),
    "scheduler": dbutils.widgets.get("scheduler"),
    "reference_date": dbutils.widgets.get("reference_date"),
    "ssrs_workspace_id": dbutils.widgets.get("ssrs_workspace_id"),
    "ssrs_report_id": dbutils.widgets.get("ssrs_report_id"),
    "additional_arguments": dbutils.widgets.get("additional_arguments"),
    "export_category": dbutils.widgets.get("export_category"),
    "redis_key": f"reports_orchestrator:reports:{dbutils.widgets.get('export_category')}:{dbutils.widgets.get('scheduler')}:{dbutils.widgets.get('report_category')}reports:{dbutils.widgets.get('report_name')}" + (f":{dbutils.widgets.get('portfolio_report')}" if dbutils.widgets.get('portfolio_report') else ""),
    "run_date": dbutils.widgets.get("run_date"),
}

inserted_params = spark.createDataFrame([params])
#display(inserted_params)



In [0]:
import json
from datetime import datetime

for row in inserted_params.collect():
    params = row.asDict()

    run_date = params.get("run_date", datetime.now().strftime("%Y-%m-%d"))
    environment = params.get("environment")
    report_category = params.get("report_category")
    report_name = params.get("report_name")
    portfolio_report = params.get("portfolio_report")
    report_job_id = params.get('job_id')
    delivery_job_id = params.get('delivery_id')
    email_job_id = params.get('email_id')
    ssrs_report_id = params.get('ssrs_report_id')
    ssrs_workspace_id = params.get('ssrs_workspace_id')
    scheduler = params.get("scheduler")
    reference_date = params.get("reference_date")
    additional_arguments = params.get("additional_arguments")
    redis_key = params.get("redis_key")

    report_output = f"/Volumes/{environment}-cdh-delivery/{report_category.lower()}reports/{report_name.lower()}/"
    destination_path = f"s3://{environment}-ec1-cepal-common-zone/onprem-outgoing/{report_category}reports/{report_name.lower()}/"

    report_name_display = f"{report_name}_{report_category}"

    report_args = {}
    if ssrs_report_id and ssrs_workspace_id and reference_date:
        report_args["ssrs_workspace_id"] = ssrs_workspace_id
        report_args["ssrs_report_id"] = ssrs_report_id    
        report_args["reference_date"] = reference_date

    report_args["portfolio_name"] = portfolio_report.upper()
    report_args["output_file_prefix"] = f"{portfolio_report.capitalize()}_{report_name.capitalize()}_report_" if portfolio_report else f"{report_name.capitalize()}_report_"
    report_args["report_output"] = report_output

    if additional_arguments:
        json_args = json.loads(additional_arguments)
        report_args["additional_arguments"] = json_args

    if portfolio_report:
        jsonname = f"{portfolio_report.upper()}_{report_name_display.upper()}"
        jsonjobname = f"{report_category.capitalize()}Reports_{report_name.capitalize()}_{portfolio_report.capitalize()}"
        jsonsubject = f"{portfolio_report.capitalize()} {report_name} Report"
        jsonentra = f"DNA_{report_category.capitalize()}Reports_{report_name.capitalize()}_{portfolio_report.capitalize()}"
        jsonerror = f"Internal Notification: {report_category.capitalize()}Reports_{report_name.capitalize()}_{portfolio_report.capitalize()} ERROR"
    else:
        jsonname = f"{report_name_display.upper()}"
        jsonjobname = f"{report_category.capitalize()}Reports_{report_name.capitalize()}"
        jsonsubject = f"{report_name} Report"
        jsonentra = f"DNA_{report_category.capitalize()}Reports_{report_name.capitalize()}"
        jsonerror = f"Internal Notification: {report_category.capitalize()}Reports_{report_name.capitalize()} ERROR"

    jsonlocation = f"\\\\dwhsideapps\\share\\in\\{environment}\\{report_category.lower()}reports\\{report_name.lower()}\\"

    report_json = {
        "report_name": jsonjobname,
        "report_job_name": jsonjobname,
        "report_schedule": "0 0 * * *",
        "report_job_id": f"{report_job_id}",
        "delivery_job_id": f"{delivery_job_id}",
        "email_job_id": f"{email_job_id}",
        "model": 1,
        "report_args": report_args,
        "delivery_args": {
            "method": "s3",
            "sftp_args": {},
            "sharepoint_args": {},
            "s3_args": {
                "destination_path": destination_path
            }
        },
        "email_args": {
            "external": {
                "recipients_to": [],
                "recipients_cc": [],
                "recipients_entra_group": jsonentra,
                "recipients_bcc": [],
                "subject": jsonsubject,
                "template_path": f"/Volumes/{environment}-cdh/reference/reportscatalog/email_templates/report_success_template_email.html",
                "report_location_info": jsonlocation
            },
            "internal": {
                "recipients_to": [
                    "dna-sys@cepal.gr"
                ],
                "recipients_cc": [
                    "extaxenakis@cepal.gr",
                    "kspanos@cepal.gr",
                    "vtsourouni@cepal.gr",
                    "gtsatsaronis@cepal.gr",
                    "cchaziraki@cepal.gr",
                    "nsatravelas@cepal.gr"
                ],
                "recipients_entra_group": "",
                "recipients_bcc": [],
                "subject": jsonerror,
                "template_path": f"/Volumes/{environment}-cdh/reference/reportscatalog/email_templates/report_failure_template_email.html"
            }
        },
        "is_report_active": "True"
    }

    #json_key = f"reports_orchestrator:reports:{scheduler}:{report_category}reports:{report_name.lower()}:{portfolio_report.lower()}" if portfolio_report else f"reports_orchestrator:reports:{scheduler}:{report_category}reports:{report_name.lower()}"


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

existing_records = f"""
        SELECT *
        FROM `{env_prefix}-cdh`.redis.reports as
        where redis_key = '{params['redis_key']}' or report_name = '{jsonjobname}'
    """
existing_records_list=[]
existing_records_df = spark.sql(existing_records)
existing_records_list = existing_records_df.collect()
#display(existing_records_list)

if existing_records_list:
    raise ValueError ("Record already exists")


schema = StructType([
    StructField("redis_key", StringType(), True),
    StructField("report_name", StringType(), True),
    StructField("isdeployed", IntegerType(), True),
    StructField("redis_json", StringType(), True)
])

report_json_df = spark.createDataFrame([{
    "redis_key": redis_key,
    "report_name": jsonjobname,
    "isdeployed": 0,
    "redis_json": report_json
}], schema=schema)

report_json_df.write.format("delta").mode("append").saveAsTable(f"`{env_prefix}-cdh`.redis.reports")
display(report_json_df)